# EXAONE 3.5 2.4B QLoRA 파인튜닝 (Colab T4)

`ft_v2_training.jsonl`의 `instruction`과 `input`을 사용자 프롬프트로, `output`을 정답으로 사용합니다. EXAONE의 chat template을 적용하고, 정답 답변 부분만 loss에 포함합니다. T4 메모리를 고려해 NF4 4-bit QLoRA, FP16 계산, micro-batch 1, gradient accumulation을 사용합니다.

로컬 데이터 점검 결과: train 160,000건, validation 20,000건. 두 파일 모두 JSONL이며 세 필드가 전부 존재하고 비어 있지 않습니다. 입력의 train/validation 중복은 0건입니다. 시작은 train 5,000건·validation 500건 파일럿이며, 전체 학습은 `MAX_TRAIN_ROWS = None`으로 바꾸세요.

EXAONE은 `trust_remote_code=True`가 필요합니다. 이전 추론 오류와 같은 호환성 문제를 피하려고 `transformers<5.9`를 고정합니다. 패키지를 설치한 뒤 런타임을 재시작하고 아래 셀부터 실행하세요.

참고: [EXAONE 모델 카드](https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct), [Hugging Face QLoRA 가이드](https://huggingface.co/docs/transformers/quantization/bitsandbytes), [PEFT LoRA 가이드](https://huggingface.co/docs/peft/package_reference/lora)

In [ ]:
%pip install -q 'transformers<5.9' peft bitsandbytes datasets accelerate sentencepiece

In [ ]:
from google.colab import drive
from pathlib import Path
from datasets import load_dataset
import torch, transformers
from packaging.version import Version

if Version(transformers.__version__) >= Version('5.9'):
    raise RuntimeError(f'transformers {transformers.__version__}가 현재 EXAONE 원격 코드와 호환되지 않습니다. 패키지 설치 후 런타임을 재시작하세요.')
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임 유형을 GPU(T4)로 바꾼 뒤 다시 실행하세요.')

drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/dontalk')

def find_data(filename):
    candidates = [
        drive_dir / filename,
        Path('/content/data/processed') / filename,
        Path('data/processed') / filename,
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f'{filename}을 찾지 못했습니다. MyDrive/dontalk 또는 /content/data/processed에 업로드하세요.')
    return path

train_path = find_data('ft_v2_training.jsonl')
valid_path = find_data('ft_v2_validation.jsonl')
train_raw = load_dataset('json', data_files=str(train_path), split='train')
valid_raw = load_dataset('json', data_files=str(valid_path), split='train')
required = ('instruction', 'input', 'output')
def valid_row(row):
    return all(isinstance(row.get(k), str) and row[k].strip() for k in required)
train_raw = train_raw.filter(valid_row)
valid_raw = valid_raw.filter(valid_row)

# T4 파일럿 크기. 전체 train 사용 시 MAX_TRAIN_ROWS만 None으로 변경
MAX_TRAIN_ROWS = 5000
MAX_VALID_ROWS = 500
train_raw = train_raw.shuffle(seed=42)
valid_raw = valid_raw.shuffle(seed=42)
if MAX_TRAIN_ROWS is not None:
    train_raw = train_raw.select(range(min(MAX_TRAIN_ROWS, len(train_raw))))
if MAX_VALID_ROWS is not None:
    valid_raw = valid_raw.select(range(min(MAX_VALID_ROWS, len(valid_raw))))
print(f'train={len(train_raw):,}, validation={len(valid_raw):,}, GPU={torch.cuda.get_device_name(0)}, transformers={transformers.__version__}')

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
MAX_SEQUENCE_LENGTH = 2048
SYSTEM_MESSAGE = (
    '당신은 한국어 금융 상담원입니다. 질문에 직접 답하고, '
    '제공된 정보에 없는 수수료나 절차는 사실처럼 단정하지 마세요.'
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def tokenize_example(row):
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': f"요청: {row['instruction']}\n\n고객 문의: {row['input']}"},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=False
    )
    full_ids = tokenizer.apply_chat_template(
        messages + [{'role': 'assistant', 'content': row['output']}],
        tokenize=True, add_generation_prompt=False, return_dict=False
    )
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError('EXAONE chat template의 prompt가 학습 대화의 prefix와 일치하지 않습니다.')
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
        'length': len(full_ids),
        'target_length': len(full_ids) - len(prompt_ids),
    }

def prepare_dataset(dataset, name):
    encoded = dataset.map(tokenize_example, remove_columns=dataset.column_names, desc=f'Tokenize {name}')
    before = len(encoded)
    encoded = encoded.filter(
        lambda row: row['length'] <= MAX_SEQUENCE_LENGTH and row['target_length'] > 0,
        desc=f'Filter {name} length',
    )
    encoded = encoded.remove_columns(['length', 'target_length'])
    print(f'{name}: {len(encoded):,}/{before:,} 사용 (길이 초과/빈 답변 {before - len(encoded):,}건 제외)')
    if len(encoded) == 0:
        raise ValueError(f'{name}에 사용할 수 있는 샘플이 없습니다.')
    return encoded

train_dataset = prepare_dataset(train_raw, 'train')
eval_dataset = prepare_dataset(valid_raw, 'validation')

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = torch.float16  # T4는 BF16 대신 FP16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, device_map='auto',
    dtype=compute_dtype, quantization_config=quant_config,
)
model.eval()

## 학습 전 기준 답변 (RAG 미사용)

파인튜닝 효과만 비교하기 위해 validation의 같은 질문을 사용하고, 기본 모델과 학습 후 모델 모두 검색 문서를 입력하지 않습니다. 아래 셀의 답변은 학습 후 비교 셀까지 변수로 유지됩니다.

In [ ]:
# 파인튜닝 전 기본 모델의 기준 답변. RAG 검색 결과는 프롬프트에 넣지 않습니다.
sample = valid_raw[0]
MAX_NEW_TOKENS = 1000
messages = [
    {'role': 'system', 'content': SYSTEM_MESSAGE},
    {'role': 'user', 'content': f"요청: {sample['instruction']}\n\n고객 문의: {sample['input']}"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors='pt'
).to(model.device)
with torch.inference_mode():
    baseline_generated = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
baseline_answer = tokenizer.decode(
    baseline_generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True
)
print('=== 질문 ===\n', sample['instruction'], sample['input'])
print('\n=== 학습 전 기본 모델 답변 ===\n', baseline_answer)
del baseline_generated, inputs
torch.cuda.empty_cache()

## QLoRA 파인튜닝

위 기준 질의 셀을 먼저 실행한 다음 실행하세요.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False
# EXAONE stores token embeddings at transformer.wte; expose that path to Transformers/PEFT.
model.transformer._input_embed_layer = 'wte'
assert model.get_input_embeddings() is model.transformer.wte, 'EXAONE 입력 임베딩 경로 확인 실패'
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules='all-linear', bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

output_dir = '/content/drive/MyDrive/dontalk/exaone35-2.4b-qlora-v2'
training_args = TrainingArguments(
    output_dir=output_dir, num_train_epochs=1,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
    lr_scheduler_type='cosine', optim='paged_adamw_8bit',
    fp16=True, bf16=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    eval_strategy='steps', eval_steps=500, save_strategy='steps',
    save_steps=500, save_total_limit=2, logging_steps=20,
    train_sampling_strategy='group_by_length',
    remove_unused_columns=False, report_to='none',
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, padding=True, label_pad_token_id=-100, pad_to_multiple_of=8
    ),
)
last_checkpoint = get_last_checkpoint(output_dir) if Path(output_dir).exists() else None
print('이어 학습:', last_checkpoint or '새 학습')
trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.save_model(output_dir)  # PEFT adapter만 저장
tokenizer.save_pretrained(output_dir)
print('저장 완료:', output_dir)

In [ ]:
# 검증 loss와 샘플 응답 확인
print(trainer.evaluate())
messages = [
    {'role': 'system', 'content': SYSTEM_MESSAGE},
    {'role': 'user', 'content': f"요청: {sample['instruction']}\n\n고객 문의: {sample['input']}"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors='pt'
).to(trainer.model.device)
trainer.model.config.use_cache = True
with torch.inference_mode():
    generated = trainer.model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
answer = tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== 질문(inst 포함) ===\n',  sample['input'])
print('\n=== 학습 전 기본 모델 답변 ===\n', baseline_answer)
print('\n=== 파인튜닝 모델 답변 ===\n', answer)
print('\n=== 검증 정답 ===\n', sample['output'])

# RAG 적용

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

dataset_path = "/content/drive/MyDrive/dontalk/rag_dataset.jsonl"
embedding_path = "/content/drive/MyDrive/dontalk/rag_embeddings.npy"

df = pd.read_json(dataset_path, lines=True)
document_embeddings = np.load(embedding_path).astype("float32")

assert len(df) == len(document_embeddings)

# 문서 임베딩 정규화
document_embeddings /= np.linalg.norm(
    document_embeddings,
    axis=1,
    keepdims=True
)

def search_documents(query, top_k=5):
    # 질의는 query prompt를 사용
    query_embedding = model.encode(
        [query],
        prompt_name="query",
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # 정규화된 벡터끼리는 내적 = cosine similarity
    scores = document_embeddings @ query_embedding

    top_indices = np.argsort(scores)[::-1][:top_k]

    result = df.iloc[top_indices].copy()
    result.insert(0, "score", scores[top_indices])

    return result.reset_index(drop=True)


model = SentenceTransformer(
    "dragonkue/snowflake-arctic-embed-l-v2.0-ko"
)

### validation 데이터들을 활용해서 비교 가능

In [ ]:
## 질의를 바꿔가면서 해보자
sample = valid_raw[0]

results = search_documents(sample['input'], top_k=5)

pd.set_option("display.max_colwidth", 500)

for i, row in results.iterrows():
    print(f"[{i + 1}위] score={row['score']:.4f}")
    print(row["text"])
    print("-" * 80)

rag_to_llm = ''

for i, row in results.iterrows():
    rag_to_llm += f"[{i + 1}위] score={row['score']:.4f}"
    rag_to_llm += row["text"]
print(rag_to_llm)

In [ ]:
# 생성 후 문답 받기
messages = [
    {'role': 'system', 'content': SYSTEM_MESSAGE},
    {'role': 'user', 'content': f"요청: {sample['instruction']}\n\n고객 문의: {sample['input']} \n RAG 데이터 : {rag_to_llm}"},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors='pt'
).to(trainer.model.device)
trainer.model.config.use_cache = True
with torch.inference_mode():
    generated = trainer.model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
answer_with_tuning = tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

with trainer.model.disable_adapter():
    with torch.inference_mode():
        base_output = trainer.model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

base_answer = tokenizer.decode(
    base_output[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print('=== 질문 ===\n', sample['instruction'], sample['input'])
print('\n=== FT + RAG ===\n', answer_with_tuning)
print('\n=== RAG ===\n', base_answer)